# Gender Coding Companies

### Inputs: 
- `"../data/derived/industry_gender_share.csv"` 
- `"../data/derived/company_profiles.csv"` 
- `"../data/occupations_{year}.txt"`

### Outputs:
- `"../data/derived/company_gender_share.csv"`
- `"../data/derived/company_gender_share.xlsx"`

  

### Purpose:

Map each company to an industry category with a known percentage of women workers. 

Some companies remain unmatched, which need to be supplemented by manual review.

In [1]:
import pandas as pd
from langchain_dartmouth.llms import ChatDartmouthCloud
from langchain_core.prompts import ChatPromptTemplate

from joblib import Parallel, delayed
from tqdm.auto import tqdm

In [12]:
industry_map = {}
for year in range(2015, 2025):
    with open(f"../data/industry_{year}.txt", 'r') as f:
        industry_map[str(year)] = f.read().splitlines()

In [29]:
industry_gender = pd.read_csv("../data/bls/bls_industry_data_2024.csv")
industries = industry_gender.industry.to_list()
industry_gender.head()

,id,industry,level,parent_id,total_employed,pct_women,pct_white,pct_black,pct_asian,pct_hispanic
0,L0_0001,"Agriculture, forestry, fishing, and hunting",0,NaN,2225.0,29.1,91.7,2.8,1.0,28.2
1,L0_0002,"Mining, quarrying, and oil and gas extraction",0,NaN,531.0,13.7,86.9,7.2,2.8,25.8
2,L0_0003,Manufacturing,0,NaN,15134.0,29.1,77.9,10.6,8.1,18.7
3,L1_0004,Durable goods manufacturing,1,L0_0003,9910.0,24.9,78.5,9.6,8.6,17.2
4,L2_0005,Nonmetallic mineral products manufacturing,2,L1_0004,400.0,18.2,86.0,9.2,2.7,21.9


In [2]:
company_profiles = pd.read_csv("../data/derived/company_profiles.csv")

In [7]:
company_profiles.head()

,id,company_name,scraped_linkedin_url,name,country_code,locations,followers,employees_in_linkedin,about,specialties,...,stock_info,get_directions_url,description,additional_info,additional_information,country_codes_array,alumni,alumni_information,website_simplified,unformatted_about
0,verizon-labs,Verizon Labs,https://www.linkedin.com/company/verizon-labs,Verizon Labs,NaN,[],5193.0,83.0,NaN,NaN,...,NaN,"[{""directions_url"":""https://www.bing.com/maps?...","Verizon Labs | 5,193 followers on LinkedIn.",NaN,Additional jobs info: Network Operations Cente...,NaN,NaN,NaN,verizon.com,NaN
1,renew,ReNew Power,https://in.linkedin.com/company/renew,ReNew,IN,"[""Commercial Block, Zone 6, Golf Course Road D...",404759.0,4832.0,ReNew is a leading decarbonisation solutions p...,"Renewable Energy, Solar Energy , Wind Energy ,...",...,NaN,"[{""directions_url"":""https://www.bing.com/maps?...","ReNew | 404,759 followers on LinkedIn. ReNew i...",NaN,"Additional jobs info: Graduate Engineer (29,65...","[""IN""]",NaN,NaN,renew.com,\n ReNew is a leading decarbonisa...
2,citi,Citibank,https://www.linkedin.com/company/citi,Citi,"US,CA,MY,PA,PE,MX,PH,HK,AE,GB,CO,IN,IL,TW","[""388 Greenwich Street New York, New York 1001...",4749962.0,195441.0,Citi's mission is to serve as a trusted partne...,"Banking, Commercial Banking, Investment Bankin...",...,NaN,"[{""directions_url"":""https://www.bing.com/maps?...","Citi | 4,749,962 followers on LinkedIn. Citi&#...",NaN,"Additional jobs info: Citi (9,127 open jobs). ...","[""US"",""CA"",""MY"",""PA"",""PE"",""MX"",""PH"",""HK"",""AE"",...",NaN,NaN,citigroup.com,\n Citi's mission is to serve as ...
3,epic1979,Epic Systems Corp,https://www.linkedin.com/company/epic1979,Epic,US,"[""1979 Milky Way Verona, WI 53593, US""]",878968.0,16306.0,Join us in our mission to help the world get w...,"healthcare, emr, ehr, phr, and software",...,NaN,"[{""directions_url"":""https://www.bing.com/maps?...","Epic | 878,968 followers on LinkedIn. ...with ...",NaN,NaN,"[""US""]",NaN,NaN,epic.com,\n Join us in our mission to help...
4,dc-energy-llc,DC Energy,https://www.linkedin.com/company/dc-energy-llc,DC Energy,US,"[""1600 Tysons Blvd Fifth Floor McLean, Virgini...",4220.0,106.0,DC Energy is a proprietary trading firm that f...,NaN,...,NaN,"[{""directions_url"":""https://www.bing.com/maps?...","DC Energy | 4,220 followers on LinkedIn. DC En...",NaN,"Additional jobs info: Analyst (694,057 open jo...","[""US""]",NaN,NaN,dc-energy.com,\n DC Energy is a proprietary tra...


In [3]:
company_profiles = company_profiles[
    [
        "id",
        "company_name",
        "about",
        "specialties",
        "organization_type",
        "industries",
        "unformatted_about",
    ]
]

In [12]:
from langchain_core.output_parsers import JsonOutputParser

In [ ]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
import os
def get_census_industry(company_profile: pd.Series, industry_set: list[str]) -> dict:
    llm = ChatDartmouthCloud(
        model_name="vertex_ai.gemini-3-flash-preview",
        max_tokens=1024,
        dartmouth_chat_api_key=os.getenv("DARTMOUTH_CHAT_API_KEY")
    )
    output_schema = """```json
{
  "type": "object",
  "properties": {
      "id": {
      "type": "string",
      "description": "Unique identifier for the organization"
    },
      "name": {
      "type": "string",
      "description": "Name of the organization"
    },
      "assessment": {
      "type": "string",
      "description": "Detailed assessment or description of the organization"
    },
      "industry": {
      "type": "string",
      "description": "The industry classification"
    }
  },
  "required": ["id", "name", "assessment", "industry"],
  "additionalProperties": false
}
```
"""

    industry_prompt = ChatPromptTemplate(
        [
            (
                "system",
                "Your task is to identify the best matching "
                "industry category from a set of options for a given company. Discuss the data "
                "given in the company profile before responding with your "
                "final decision with a valid JSON object using the following schema:"
                f"\n{output_schema}\n"
                "Your assessment should be as specific to the company as possible. "
                "For example, if the company is a subsidiary of some other company, "
                "match the industry based on the subsidiary's industry, not the parent company's. "
                "If none of the provided options are a good fit, label it as N/A."
                "The available industries are:\n\n{{industries}}."
                "You MUST select your industry value verbatim from this list. "
                "Do not use any other classification scheme. "
                "If none of the above fit, use exactly 'N/A'."
            ),
            ("human", "Here is the company profile: \n\n {{company_profile}}"),
        ],
        template_format="jinja2",
    )
    industry_mapper = industry_prompt | llm | JsonOutputParser()

    return industry_mapper.invoke(
        input={
            "industries": "\n".join(f"- {ind}" for ind in industry_set),
            "company_profile": company_profile.to_json(),
        }
    )

In [58]:
import json
from pathlib import Path
from joblib import Parallel, delayed


def process_single_company(idx, company_profile, results_dir, industry_set):
    """Process one company and save result immediately"""
    result_file = Path(results_dir) / f"result_{idx}.json"

    # Skip if already processed
    if result_file.exists():
        print(f"Skipping {idx} (already processed)")
        return {"idx": idx, "status": "skipped"}

    try:
        response = get_census_industry(company_profile, industry_set)

        # Save immediately
        with open(result_file, "w") as f:
            json.dump({"idx": idx, "result": response}, f, indent=2)

        return {"idx": idx, "status": "success", "result": response}

    except Exception as e:
        # Save error info
        error_file = Path(results_dir) / f"error_{idx}.json"
        with open(error_file, "w") as f:
            json.dump({"idx": idx, "error": str(e)}, f, indent=2)

        return {"idx": idx, "status": "failed", "error": str(e)}


def process_parallel_with_saves(company_profiles, industry_set, results_dir="results/old_industries"):
    """Process in parallel with individual saves"""
    Path(results_dir).mkdir(exist_ok=True)

    company_data = [(idx, row) for idx, row in company_profiles.iterrows()]

    # Process in parallel, each saving its own result
    results = Parallel(n_jobs=-1)(
        delayed(process_single_company)(idx, company_profile, results_dir, industry_set)
        for idx, company_profile in tqdm(company_data, desc="Processing companies")
    )

    # Summarize results
    successful = [r for r in results if r["status"] == "success"]
    failed = [r for r in results if r["status"] == "failed"]
    skipped = [r for r in results if r["status"] == "skipped"]

    print(f"✅ Successful: {len(successful)}")
    print(f"❌ Failed: {len(failed)}")
    print(f"⏭️ Skipped: {len(skipped)}")

    return results


# results = process_parallel_with_saves(company_profiles)

In [ ]:
results_map = {}
for year in range(2015, 2025):
    results_map[str(year)] = process_parallel_with_saves(company_profiles, industries=industry_map[str(year)], results_dir=f"results/linkedin_map_{year}")

In [12]:
results = pd.DataFrame.from_records(
    [json.load(file.open()) for file in sorted(Path("results").glob("result_*.json"))]
)
results = pd.json_normalize(results.result)
results

,id,name,assessment,industry
0,verizon-labs,Verizon Labs,Verizon Labs is the technology innovation and ...,Computer and mathematical occupations
1,renew,ReNew Power,ReNew Power is a major clean energy provider s...,"Installation, maintenance, and repair occupations"
2,trillium-health-partners,Trillium Health Partners (Miss,Trillium Health Partners is a large-scale hosp...,Healthcare practitioners and technical occupat...
3,airforcereserverecruiting,United States Air Force Reserve / 78th Attack ...,The Air Force Reserve is a military component ...,Protective service occupations
4,sap,SAP,SAP is a global enterprise software company sp...,Computer and mathematical occupations
...,...,...,...,...
3156,nestle-s-a-,Nestle USA,Nestle USA is a major subsidiary of the global...,Production occupations
3157,kpmgindia,KPMG India Services Limited,KPMG India Services Limited is a professional ...,"Management, business, and financial operations..."
3158,paralosenergy,Para Nos Energy,Paralos Energy is a specialized firm focused o...,Construction and extraction occupations
3159,us-army-forces-command-forscom,United States Army - 1st Cavalry Division Sust...,The organization is a major command of the Uni...,Protective service occupations


In [59]:
industry_gender = pd.read_csv("../data/derived/industry_gender_share.csv")
industries = industry_gender.industry.to_list()


In [60]:
industries

['Accounting',
 'Administration of Economic Programs',
 'Agricultural Machinery Manufacturing',
 'Airlines',
 'Apparel & Fashion',
 'Appliance Manufacturing',
 'Architecture & Planning',
 'Artificial Intelligence',
 'Artists, Spectator Sports & Related Industries',
 'Automotive',
 'Automotive Rental',
 'Aviation & Aerospace',
 'Beer, Wine & Liquor Stores',
 'Beverage Manufacturing',
 'Biotechnology',
 'Book Stores',
 'Building Materials',
 'Business, Technical & Trade Schools',
 'Chemicals',
 'Civil Engineering',
 'Clothing Stores',
 'Colleges & Professional Schools',
 'Commercial & Service Industry Machinery Manufacturing',
 'Commercial Banking',
 'Commercial Equipment',
 'Commercial Real Estate',
 'Computer Design',
 'Computer Games',
 'Computer Hardware',
 'Computer Networking',
 'Computer Software',
 'Construction',
 'Construction & Mining Equipment Manufacturer',
 'Consumer Electronics',
 'Consumer Goods',
 'Defense & Space',
 'Department Stores',
 'Design Services',
 'Ecommerce',

In [62]:
results = process_parallel_with_saves(company_profiles, industry_set=industries, results_dir=f"results/jen_based")

Processing companies:   0%|          | 0/3168 [00:00<?, ?it/s]

Traceback (most recent call last):
  File "/Users/f006p17/.local/share/uv/python/cpython-3.13.12-macos-aarch64-none/lib/python3.13/multiprocessing/resource_tracker.py", line 371, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /var/folders/rt/xv7r5qdj52742frmp1y_56940000gp/T/joblib_memmapping_folder_1933_310830fef4454242acdb189d49d52231_071dafd66e374049997e528872f2fdf4 for automatic cleanup: unknown resource type folder
Traceback (most recent call last):
  File "/Users/f006p17/.local/share/uv/python/cpython-3.13.12-macos-aarch64-none/lib/python3.13/multiprocessing/resource_tracker.py", line 371, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /var/folders/rt/xv7r5qdj52742frmp1y_56940000gp/T/joblib_memmapping_folder_1933_ee7898673db241e7bd5b19c467f63e3c_5fd80a1b8109460c8

✅ Successful: 0
❌ Failed: 3168
⏭️ Skipped: 0


In [55]:
results = pd.DataFrame.from_records(
    [json.load(file.open()) for file in sorted(Path("results").glob("result_*.json"))]
)
results = pd.json_normalize(results.result)
results

,id,name,assessment,industry
0,verizon-labs,Verizon Labs,Verizon Labs is the technology innovation and ...,Computer and mathematical occupations
1,renew,ReNew Power,ReNew Power is a major clean energy provider s...,"Installation, maintenance, and repair occupations"
2,trillium-health-partners,Trillium Health Partners (Miss,Trillium Health Partners is a large-scale hosp...,Healthcare practitioners and technical occupat...
3,airforcereserverecruiting,United States Air Force Reserve / 78th Attack ...,The Air Force Reserve is a military component ...,Protective service occupations
4,sap,SAP,SAP is a global enterprise software company sp...,Computer and mathematical occupations
...,...,...,...,...
3163,nestle-s-a-,Nestle USA,Nestle USA is a major subsidiary of the global...,Production occupations
3164,kpmgindia,KPMG India Services Limited,KPMG India Services Limited is a professional ...,"Management, business, and financial operations..."
3165,paralosenergy,Para Nos Energy,Paralos Energy is a specialized firm focused o...,Construction and extraction occupations
3166,us-army-forces-command-forscom,United States Army - 1st Cavalry Division Sust...,The organization is a major command of the Uni...,Protective service occupations


In [56]:
results.head().to_csv("result_sample.csv", index=False)

In [20]:
def clean_dataframe(df):
    df_cleaned = df.copy()

    # Get all prefixed columns
    prefixed_cols = [col for col in df.columns if col.startswith("properties.")]

    for prefixed_col in prefixed_cols:
        base_col = prefixed_col.replace("properties.", "", 1)

        if base_col in df.columns:
            # Merge the columns
            df_cleaned[base_col] = df_cleaned[base_col].fillna(df_cleaned[prefixed_col])
            # Drop the prefixed column
            df_cleaned = df_cleaned.drop(columns=[prefixed_col])

    return df_cleaned


# Usage
df_cleaned = clean_dataframe(results)

In [21]:
df_cleaned

,id,name,assessment,industry
0,verizon-labs,Verizon Labs,Verizon Labs is the technology innovation and ...,Computer and mathematical occupations
1,renew,ReNew Power,ReNew Power is a major clean energy provider s...,"Installation, maintenance, and repair occupations"
2,trillium-health-partners,Trillium Health Partners (Miss,Trillium Health Partners is a large-scale hosp...,Healthcare practitioners and technical occupat...
3,airforcereserverecruiting,United States Air Force Reserve / 78th Attack ...,The Air Force Reserve is a military component ...,Protective service occupations
4,sap,SAP,SAP is a global enterprise software company sp...,Computer and mathematical occupations
...,...,...,...,...
3163,nestle-s-a-,Nestle USA,Nestle USA is a major subsidiary of the global...,Production occupations
3164,kpmgindia,KPMG India Services Limited,KPMG India Services Limited is a professional ...,"Management, business, and financial operations..."
3165,paralosenergy,Para Nos Energy,Paralos Energy is a specialized firm focused o...,Construction and extraction occupations
3166,us-army-forces-command-forscom,United States Army - 1st Cavalry Division Sust...,The organization is a major command of the Uni...,Protective service occupations


In [22]:
df_cleaned = df_cleaned.dropna(subset="id")

After cleaning, a few companies still need mapping. Re-run in a loop until all have been mapped.

In [54]:
def map_remaining_companies(already_mapped, industries, remaining_dir):
    import shutil

    try:
        shutil.rmtree(remaining_dir)
    except FileNotFoundError:
        pass

    remaining_profiles = company_profiles[
        ~company_profiles["id"].isin(already_mapped["id"])
    ]
    results = process_parallel_with_saves(
        remaining_profiles, industries, results_dir=remaining_dir
    )
    results = pd.DataFrame.from_records(
        [
            json.load(file.open())
            for file in sorted(Path("results/remaining/").glob("result_*.json"))
        ]
    )
    results = pd.json_normalize(results.result)
    remaining_df_cleaned = clean_dataframe(results)
    remaining_df_cleaned = remaining_df_cleaned.dropna(subset="id")
    df = pd.concat([already_mapped, remaining_df_cleaned])
    return df

Traceback (most recent call last):
  File "/Users/f006p17/.local/share/uv/python/cpython-3.13.12-macos-aarch64-none/lib/python3.13/multiprocessing/resource_tracker.py", line 371, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /loky-1933-t_rc3s93 for automatic cleanup: unknown resource type semlock
Traceback (most recent call last):
  File "/Users/f006p17/.local/share/uv/python/cpython-3.13.12-macos-aarch64-none/lib/python3.13/multiprocessing/resource_tracker.py", line 371, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /loky-1933-b4jdds9j for automatic cleanup: unknown resource type semlock
Traceback (most recent call last):
  File "/Users/f006p17/.local/share/uv/python/cpython-3.13.12-macos-aarch64-none/lib/python3.13/multiprocessing/resource_tracker.py", line 371, i

In [ ]:
while df_cleaned.shape[0] < company_profiles.shape[0]:
    df_cleaned = map_remaining_companies(df_cleaned, industries, "results/jen_based/remaining")

Processing companies:   0%|          | 0/15 [00:00<?, ?it/s]

Traceback (most recent call last):
  File "/Users/f006p17/.local/share/uv/python/cpython-3.13.12-macos-aarch64-none/lib/python3.13/multiprocessing/resource_tracker.py", line 371, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /var/folders/rt/xv7r5qdj52742frmp1y_56940000gp/T/joblib_memmapping_folder_1933_101bc04a1cda4949aa859355c50b16a0_d7a82456d44b4e3cba9e2ce6aa95fb1f for automatic cleanup: unknown resource type folder
Traceback (most recent call last):
  File "/Users/f006p17/.local/share/uv/python/cpython-3.13.12-macos-aarch64-none/lib/python3.13/multiprocessing/resource_tracker.py", line 371, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /var/folders/rt/xv7r5qdj52742frmp1y_56940000gp/T/joblib_memmapping_folder_1933_ee7898673db241e7bd5b19c467f63e3c_da37dad78b394ee6a

✅ Successful: 15
❌ Failed: 0
⏭️ Skipped: 0


In [31]:
df_cleaned = df_cleaned[["id", "name", "assessment", "industry"]]

In [32]:
if "occupation" in [c for c in industry_gender.columns]:
    industry_gender.rename(columns={'occupation': 'industry'}, inplace=True)

In [33]:
if "pct_women" in [c for c in industry_gender.columns]:
    industry_gender.rename(columns={'pct_women': 'women_pct'}, inplace=True)

In [47]:
industry_gender = industry_gender.set_index("industry")["women_pct"].to_dict()

In [48]:
industry_gender

{'Accounting': 60.9,
 'Administration of Economic Programs': 43.8,
 'Agricultural Machinery Manufacturing': 20.7,
 'Airlines': 41.2,
 'Apparel & Fashion': 53.2,
 'Appliance Manufacturing': 23.7,
 'Architecture & Planning': 25.9,
 'Artificial Intelligence': 9.1,
 'Artists, Spectator Sports & Related Industries': 43.7,
 'Automotive': 17.9,
 'Automotive Rental': 34.4,
 'Aviation & Aerospace': 29.6,
 'Beer, Wine & Liquor Stores': 39.8,
 'Beverage Manufacturing': 25.4,
 'Biotechnology': 44.9,
 'Book Stores': 60.1,
 'Building Materials': 30.1,
 'Business, Technical & Trade Schools': 59.5,
 'Chemicals': 37.3,
 'Civil Engineering': 17.1,
 'Clothing Stores': 73.2,
 'Colleges & Professional Schools': 55.3,
 'Commercial & Service Industry Machinery Manufacturing': 25.8,
 'Commercial Banking': 38.2,
 'Commercial Equipment': 32.7,
 'Commercial Real Estate': 37.0,
 'Computer Design': 27.8,
 'Computer Games': 30.0,
 'Computer Hardware': 15.7,
 'Computer Networking': 14.9,
 'Computer Software': 37.1,


In [49]:
df_cleaned.loc[:, "women_pct"] = df_cleaned.industry.map(industry_gender)

In [50]:
df_cleaned

,id,name,assessment,industry,women_pct
0,verizon-labs,Verizon Labs,Verizon Labs is the technology innovation and ...,Computer and mathematical occupations,NaN
1,renew,ReNew Power,ReNew Power is a major clean energy provider s...,"Installation, maintenance, and repair occupations",NaN
2,trillium-health-partners,Trillium Health Partners (Miss,Trillium Health Partners is a large-scale hosp...,Healthcare practitioners and technical occupat...,NaN
3,airforcereserverecruiting,United States Air Force Reserve / 78th Attack ...,The Air Force Reserve is a military component ...,Protective service occupations,NaN
4,sap,SAP,SAP is a global enterprise software company sp...,Computer and mathematical occupations,NaN
...,...,...,...,...,...
10,null,Sanofi Genzyme,Sanofi Genzyme is the specialty care division ...,Biotechnology,44.9
11,null,Tufts University Varsity Swim Team,The Tufts University Varsity Swim Team is a co...,Colleges & Professional Schools,55.3
12,null,University of Texas - Dallas,The University of Texas at Dallas is a promine...,Colleges & Professional Schools,55.3
13,null,Unknown,The company profile contains no descriptive da...,N/A,NaN


In [37]:
df_cleaned.to_csv("../data/derived/company_gender_share_jen.csv", index=False)
df_cleaned.to_excel("../data/derived/company_gender_share_jen.xlsx", index=False)

We can get some sense of accuracy by checking the hand-labeled industries against the automatically-labeled ones:

In [38]:
reference = (
    pd.read_excel(
        "../data/supplemental/indus_gender_forsimon_cleaned.xlsx", na_values=["."]
    )[["employer", "indus_Simon", "indus_gen_percentage"]]
    .dropna()
    .drop_duplicates()
)

In [39]:
overlap = df_cleaned.merge(reference, how="inner", left_on="name", right_on="employer")

In [41]:
overlap

,id,name,assessment,industry,women_pct,employer,indus_Simon,indus_gen_percentage
0,renew,ReNew Power,ReNew Power is a major clean energy provider s...,"Installation, maintenance, and repair occupations",NaN,ReNew Power,Renewables & Environment,32.0
1,pepsico,PepsiCo,PepsiCo is a global food and beverage corporat...,"Production, transportation, and material movin...",NaN,PepsiCo,Food & Beverages,41.5
2,intel-corporation,Intel Corporation,Intel Corporation is a global technology leade...,Production occupations,NaN,Intel Corporation,Semiconductors,14.9
3,digitas-north-america,Digitas,Digitas is a global marketing and technology a...,"Arts, design, entertainment, sports, and media...",NaN,Digitas,"Marketing, Advertising & Public Relations",51.3
4,jpmorganchase,J.P. Morgan Chase & Co.,J.P. Morgan Chase & Co. is a global leader in ...,Business and financial operations occupations,NaN,J.P. Morgan Chase & Co.,Financial Services,38.2
...,...,...,...,...,...,...,...,...
168,mercer-management-consulting,Oliver Wyman,Oliver Wyman is a global management consulting...,Management occupations,NaN,Oliver Wyman,Management Consulting,42.0
169,keybanc-capital-markets,KeyBanc Capital Markets,KeyBanc Capital Markets is an investment banki...,Business and financial operations occupations,NaN,KeyBanc Capital Markets,Investment Banking,23.8
170,bechtel-corporation,Bechtel Corporation,"Bechtel Corporation is a global engineering, c...",Construction and extraction occupations,NaN,Bechtel Corporation,Construction,10.3
171,credicorpcapital,Credicorp Capital,Credicorp Capital is an investment management ...,Business and financial operations occupations,NaN,Credicorp Capital,Financial Services,38.2


In [ ]:
overlap.to_excel(
    "../data/derived/company_industry_mapping_ai_vs_human.xlsx", index=False
)